# Analise Macroeconomica — Brasil

Analise do cenario macroeconomico brasileiro com dados reais do BCB, IBGE e analyzers.

Fontes:
- **BCBFetcher**: Selic, CDI, IPCA, IGP-M, PTAX, TR, INPC, poupanca
- **IBGEFetcher**: PIB, desemprego, producao industrial
- **FiscalAnalyzer**: divida/PIB, resultado primario, trajetoria fiscal
- **CurrencyAnalyzer**: USD/BRL, carry trade, cambio real efetivo

**Nota**: Este notebook faz chamadas reais a APIs publicas (BCB, IBGE). Nao requer API keys.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 14

## 1. Politica Monetaria — Selic e CDI

A taxa Selic e a principal ferramenta de politica monetaria do BCB.
O CDI acompanha de perto a Selic e serve como benchmark para renda fixa.

In [ ]:
from carteira_auto.data.fetchers import BCBFetcher

bcb = BCBFetcher()

# Selic meta — serie historica de 5 anos
selic = bcb.get_selic(period_days=5 * 365)
print(f"Selic atual: {selic['valor'].iloc[-1]:.2f}% a.a.")
print(f"Periodo: {selic['data'].iloc[0].date()} a {selic['data'].iloc[-1].date()}")
print(f"Registros: {len(selic)}")

In [ ]:
# CDI
cdi = bcb.get_cdi(period_days=5 * 365)
print(f"CDI atual: {cdi['valor'].iloc[-1]:.4f}% a.d.")

# Visualizar Selic
fig, ax = plt.subplots()
ax.plot(selic["data"], selic["valor"], linewidth=2, color="#1f77b4")
ax.set_title("Taxa Selic Meta — Ultimos 5 Anos")
ax.set_ylabel("% a.a.")
ax.axhline(y=selic["valor"].iloc[-1], color="red", linestyle="--", alpha=0.5,
           label=f"Atual: {selic['valor'].iloc[-1]:.2f}%")
ax.legend()
plt.tight_layout()
plt.show()

## 2. Inflacao — IPCA e IGP-M

In [ ]:
# IPCA mensal
ipca = bcb.get_ipca(period_days=3 * 365)
igpm = bcb.get_igpm(period_days=3 * 365)

print(f"IPCA ultimo mes: {ipca['valor'].iloc[-1]:.2f}%")
print(f"IGP-M ultimo mes: {igpm['valor'].iloc[-1]:.2f}%")

# IPCA acumulado 12 meses
ipca_12m = ipca.tail(12)
ipca_acum = ((1 + ipca_12m["valor"] / 100).prod() - 1) * 100
print(f"\nIPCA acumulado 12m: {ipca_acum:.2f}%")

In [ ]:
# Visualizar IPCA mensal
fig, ax = plt.subplots()
colors = ["#d62728" if v > 0 else "#2ca02c" for v in ipca["valor"]]
ax.bar(ipca["data"], ipca["valor"], color=colors, alpha=0.7)
ax.set_title("IPCA Mensal — Ultimos 3 Anos")
ax.set_ylabel("% mensal")
ax.axhline(y=0, color="black", linewidth=0.5)
plt.tight_layout()
plt.show()

## 3. Cambio — USD/BRL e PTAX

In [ ]:
# PTAX compra (serie de 2 anos)
ptax = bcb.get_ptax(period_days=2 * 365)
print(f"PTAX atual: R$ {ptax['valor'].iloc[-1]:.4f}")

# Variacao 12m
if len(ptax) >= 252:
    var_12m = (ptax["valor"].iloc[-1] / ptax["valor"].iloc[-252] - 1) * 100
    print(f"Variacao 12m: {var_12m:+.1f}%")

# Visualizar
fig, ax = plt.subplots()
ax.plot(ptax["data"], ptax["valor"], linewidth=1.5, color="#2ca02c")
ax.set_title("PTAX Compra (USD/BRL) — Ultimos 2 Anos")
ax.set_ylabel("R$/USD")
ax.fill_between(ptax["data"], ptax["valor"], alpha=0.1, color="green")
plt.tight_layout()
plt.show()

## 4. Atividade Economica — PIB e Desemprego

In [ ]:
from carteira_auto.data.fetchers import IBGEFetcher

ibge = IBGEFetcher()

# PIB trimestral
pib = ibge.get_pib(quarters=12)
print(f"PIB: {len(pib)} trimestres")
print(pib.tail(4))

In [ ]:
# Desemprego (PNAD Continua)
desemprego = ibge.get_unemployment(quarters=12)
print(f"Desemprego: {len(desemprego)} trimestres")
print(desemprego.tail(4))

## 5. Juros Reais — Selic vs IPCA

In [ ]:
# Juro real = Selic - IPCA acum. 12m (aproximacao)
selic_atual = selic["valor"].iloc[-1]
juro_real = selic_atual - ipca_acum

print(f"Selic: {selic_atual:.2f}% a.a.")
print(f"IPCA 12m: {ipca_acum:.2f}%")
print(f"Juro real aproximado: {juro_real:.2f}% a.a.")
print(f"\nInterpretacao: {'Restritivo' if juro_real > 3 else 'Neutro' if juro_real > 0 else 'Expansionista'}")

## 6. Analise Fiscal — FiscalAnalyzer

O FiscalAnalyzer busca 5 series do BCB e classifica a trajetoria fiscal
com gradacao: stable → warning → critical → severe.

In [ ]:
from carteira_auto.analyzers import FiscalAnalyzer
from carteira_auto.core.engine import PipelineContext

fiscal = FiscalAnalyzer()
ctx = PipelineContext()
ctx = fiscal.run(ctx)

fm = ctx["fiscal_metrics"]
print("=== Metricas Fiscais ===")
print(f"Divida bruta/PIB: {fm.divida_bruta_pib}%")
print(f"Divida liquida/PIB: {fm.divida_liquida_pib}%")
print(f"Resultado primario/PIB: {fm.resultado_primario_pib}%")
print(f"Juros nominais/PIB: {fm.juros_nominais_pib}%")
print(f"Variacao divida 12m: {fm.divida_bruta_pib_change_12m} pp")
print(f"Trajetoria: {fm.fiscal_trajectory}")
print(f"\nSumario: {fm.summary}")

## 7. Analise Cambial — CurrencyAnalyzer

O CurrencyAnalyzer busca PTAX, DXY, Selic e Fed Funds
para calcular carry spread e tendencia cambial.

**Nota**: requer `FRED_API_KEY` para Fed Funds Rate. Sem a key, carry_spread sera None.

In [ ]:
from carteira_auto.analyzers import CurrencyAnalyzer

currency = CurrencyAnalyzer()
ctx = PipelineContext()
ctx = currency.run(ctx)

cm = ctx["currency_metrics"]
print("=== Metricas Cambiais ===")
print(f"USD/BRL (PTAX): R$ {cm.usd_brl}")
print(f"Variacao 1m: {cm.usd_brl_change_1m}%")
print(f"Variacao 3m: {cm.usd_brl_change_3m}%")
print(f"Variacao 12m: {cm.usd_brl_change_12m}%")
print(f"DXY: {cm.dxy}")
print(f"Selic: {cm.selic_rate}% | Fed Funds: {cm.fed_funds_rate}%")
print(f"Carry spread: {cm.carry_spread} pp")
print(f"Cambio real efetivo: {cm.taxa_cambio_real_efetiva}")
print(f"\nSumario: {cm.summary}")

## 8. MacroAnalyzer — Contexto Consolidado

O MacroAnalyzer consolida 11 indicadores macro em um unico `MacroContext`.

In [ ]:
from carteira_auto.analyzers import MacroAnalyzer

macro = MacroAnalyzer()
ctx = PipelineContext()
ctx = macro.run(ctx)

mc = ctx["macro_context"]
print("=== Contexto Macro Consolidado ===")
print(f"Selic: {mc.selic}% | CDI: {mc.cdi}%")
print(f"IPCA: {mc.ipca}% | IGP-M: {mc.igpm}%")
print(f"PTAX: R$ {mc.ptax_compra}")
print(f"Poupanca: {mc.poupanca}% | TR: {mc.tr}%")
print(f"PIB crescimento: {mc.pib_growth}")
print(f"Desocupacao: {mc.unemployment}%")
print(f"\nSumario: {mc.summary}")

# Erros parciais (se houve falhas)
if ctx.has_errors:
    print(f"\nErros parciais: {ctx.errors}")

## Resumo do Cenario Macro

Os dados acima permitem avaliar:

| Dimensao | Indicadores | Analise |
|----------|-------------|---------|
| **Politica monetaria** | Selic, CDI, juro real | Restritivo se juro real > 3% |
| **Inflacao** | IPCA, IGP-M | Dentro da meta? Tendencia? |
| **Cambio** | PTAX, DXY, carry | Apreciacao ou depreciacao do BRL? |
| **Atividade** | PIB, desemprego | Ciclo de expansao ou contracao? |
| **Fiscal** | Divida/PIB, primario | Trajetoria sustentavel? |